In [ ]:
import pandas as pd
import re 
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stopwords = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # lowercasing
    lowercased_text = text.lower()

    # cleaning 
    remove_punctuation = re.sub(r'[^\w\s]', '', lowercased_text)
    remove_white_space = remove_punctuation.strip()

    # Tokenization = Breaking down each sentence into an array
    tokenized_text = word_tokenize(remove_white_space)

    # Stop Words/filtering = Removing irrelevant words
    stopwords_removed = [word for word in tokenized_text if word not in stopwords]

    # Stemming = Transforming words into their base form
    stemmed_text = [stemmer.stem(word) for word in stopwords_removed]
    
    return ' '.join(stemmed_text)

texts = []
with open('/home/user/kew/projects/specific_hospo_respo/data/hotel/500k/train.response', 'r', encoding='utf8') as f:
    for line in f:
        texts.append(line.strip())
        
stemmed_texts = [preprocess_text(text) for text in texts]
df = pd.DataFrame({
    'text': texts,
    'stemmed': stemmed_texts
}
)
df.head()

KeyboardInterrupt: 

In [ ]:
from collections import Counter
import numpy as np
from scipy.sparse import csr_matrix, coo_matrix
from typing import List, Tuple
import re
from math import log

def compute_tfidf_vectors_efficient(documents: List[str], 
                                  min_df: float = 0.001,
                                  max_df: float = 0.95,
                                  max_features: int = 50000) -> Tuple[csr_matrix, List[str]]:
    """
    Compute TF-IDF vectors for a corpus of documents using sparse matrices.
    
    Args:
        documents: List of strings, where each string is a document
        min_df: Minimum document frequency (fraction) for a term to be included
        max_df: Maximum document frequency (fraction) for a term to be included
        max_features: Maximum number of features to keep
        
    Returns:
        tuple containing:
        - sparse matrix of TF-IDF vectors (num_docs x num_terms)
        - vocabulary list corresponding to matrix columns
    """
    # Step 1: Initial vocabulary building and document frequency counting
    word_doc_freq = Counter()
    doc_word_counts = []
    
    print("Processing documents...")
    for i, doc in enumerate(documents):
        if i % 10000 == 0:
            print(f"Processing document {i}/{len(documents)}")
            
        # Basic preprocessing: lowercase and split
        words = re.findall(r'\w+', doc.lower())
        
        # Store word counts for this document
        doc_counts = Counter(words)
        doc_word_counts.append(doc_counts)
        
        # Update document frequencies
        word_doc_freq.update(set(doc_counts.keys()))
    
    # Step 2: Filter vocabulary based on document frequency
    n_docs = len(documents)
    min_docs = max(1, int(min_df * n_docs))
    max_docs = int(max_df * n_docs)
    
    # Filter terms by document frequency and limit features
    valid_terms = {
        term for term, freq in word_doc_freq.most_common()
        if min_docs <= freq <= max_docs
    }
    
    if len(valid_terms) > max_features:
        valid_terms = {term for term, _ in 
                      word_doc_freq.most_common(max_features)}
    
    # Create vocabulary mapping
    vocabulary = sorted(valid_terms)
    term_to_id = {term: i for i, term in enumerate(vocabulary)}
    
    # Step 3: Build sparse TF-IDF matrix
    rows, cols, data = [], [], []
    
    print("Building sparse matrix...")
    for doc_idx, doc_counts in enumerate(doc_word_counts):
        if doc_idx % 10000 == 0:
            print(f"Processing document {doc_idx}/{len(documents)}")
            
        # Get document length for TF normalization
        doc_length = sum(count for term, count in doc_counts.items()
                        if term in term_to_id)
        
        if doc_length == 0:
            continue
            
        # Add entries for each term in vocabulary
        for term, count in doc_counts.items():
            if term in term_to_id:
                rows.append(doc_idx)
                cols.append(term_to_id[term])
                data.append(count / doc_length)  # TF normalization
    
    # Create sparse TF matrix
    tf_matrix = csr_matrix((data, (rows, cols)), 
                          shape=(len(documents), len(vocabulary)))
    
    # Compute IDF scores
    doc_frequencies = np.bincount(cols, minlength=len(vocabulary))
    idf = np.log((n_docs + 1) / (doc_frequencies + 1)) + 1
    
    # Multiply TF matrix by IDF scores and ensure result is CSR format
    tfidf_matrix = tf_matrix.multiply(idf).tocsr()
    
    return tfidf_matrix, vocabulary

def get_top_terms_efficient(doc_idx: int, 
                          tfidf_matrix: csr_matrix, 
                          vocabulary: List[str], 
                          n: int = 5) -> List[Tuple[str, float]]:
    """
    Get the top N terms with highest TF-IDF scores for a given document.
    """
    # Ensure matrix is in CSR format
    if not isinstance(tfidf_matrix, csr_matrix):
        tfidf_matrix = tfidf_matrix.tocsr()
    
    # Get the document vector
    doc_vector = tfidf_matrix[doc_idx]
    
    # Convert to array only for this single document
    scores = doc_vector.toarray().flatten()
    
    # Get top indices
    top_indices = scores.argsort()[-n:][::-1]
    
    # Return term-score pairs
    return [(vocabulary[idx], scores[idx]) 
            for idx in top_indices 
            if scores[idx] > 0]

def print_matrix_stats(tfidf_matrix: csr_matrix, vocabulary: List[str]):
    """
    Print useful statistics about the TF-IDF matrix.
    """
    print(f"Matrix shape: {tfidf_matrix.shape}")
    print(f"Vocabulary size: {len(vocabulary)}")
    print(f"Matrix density: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4%}")
    print(f"Memory usage: {tfidf_matrix.data.nbytes / 1024 / 1024:.2f} MB (data array only)")

In [11]:
infile = "/srv/scratch1/kew/bart/hospo_respo/en/data/hotel/500k/train.response"
documents = []
with open(infile, "r", encoding='utf8') as f:
    for line in f:
        documents.append(line.strip())
print(len(documents))


tfidf_matrix, vocabulary = compute_tfidf_vectors_efficient(
    documents,
    min_df=0.001,        # Term must appear in at least 0.1% of documents
    max_df=0.95,         # Term must not appear in more than 95% of documents
    max_features=50000   # Keep only top 50k terms
)

# Get top terms for a document
top_terms = get_top_terms_efficient(0, tfidf_matrix, vocabulary, n=5)
print(f"Top terms: {top_terms}")

# Print matrix info
print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"Matrix density: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4%}")

450367
Processing documents...
Processing document 0/450367
Processing document 10000/450367
Processing document 20000/450367
Processing document 30000/450367
Processing document 40000/450367
Processing document 50000/450367
Processing document 60000/450367
Processing document 70000/450367
Processing document 80000/450367
Processing document 90000/450367
Processing document 100000/450367
Processing document 110000/450367
Processing document 120000/450367
Processing document 130000/450367
Processing document 140000/450367
Processing document 150000/450367
Processing document 160000/450367
Processing document 170000/450367
Processing document 180000/450367
Processing document 190000/450367
Processing document 200000/450367
Processing document 210000/450367
Processing document 220000/450367
Processing document 230000/450367
Processing document 240000/450367
Processing document 250000/450367
Processing document 260000/450367
Processing document 270000/450367
Processing document 280000/4503

In [15]:
def compute_document_uniqueness(doc_idx: int, 
                              tfidf_matrix: csr_matrix,
                              percentile_threshold: float = 95) -> float:
    """
    Compute a uniqueness score for a document relative to the corpus.
    
    The score is based on how many terms in the document have unusually high TF-IDF
    scores compared to the corpus average. A higher score indicates a more unique document.
    
    Args:
        doc_idx: Index of the document to analyze
        tfidf_matrix: TF-IDF matrix in CSR format
        percentile_threshold: Percentile threshold for considering a term score as "high"
        
    Returns:
        float: Uniqueness score between 0 and 1, where higher values indicate more unique documents
    """
    # Ensure matrix is in CSR format
    if not isinstance(tfidf_matrix, csr_matrix):
        tfidf_matrix = tfidf_matrix.tocsr()
    
    # Get the document's vector
    doc_vector = tfidf_matrix[doc_idx].toarray().flatten()
    
    # Get non-zero terms in the document
    doc_terms = doc_vector > 0
    
    if not np.any(doc_terms):
        return 0.0  # Empty document
    
    # Calculate column-wise (term-wise) means and standard deviations
    # Note: We use mean of non-zero values to avoid bias from sparse matrix
    term_stats = []
    for j in range(tfidf_matrix.shape[1]):
        col = tfidf_matrix.getcol(j).data
        if len(col) > 0:  # Only consider terms that appear in at least one document
            term_stats.append({
                'mean': np.mean(col),
                'std': np.std(col) if len(col) > 1 else 0,
                'percentile': np.percentile(col, percentile_threshold)
            })
        else:
            term_stats.append({'mean': 0, 'std': 0, 'percentile': 0})
    
    # Count how many terms in the document have unusually high TF-IDF scores
    unusual_terms = 0
    total_terms = 0
    
    for term_idx, term_value in enumerate(doc_vector):
        if term_value > 0:  # Only consider terms present in the document
            total_terms += 1
            if term_value > term_stats[term_idx]['percentile']:
                unusual_terms += 1
    
    # Compute uniqueness score as the proportion of unusual terms
    uniqueness_score = unusual_terms / total_terms if total_terms > 0 else 0.0
    
    return uniqueness_score

def get_corpus_uniqueness_stats(tfidf_matrix: csr_matrix, 
                              percentile_threshold: float = 95) -> dict:
    """
    Compute uniqueness statistics for the entire corpus.
    
    Args:
        tfidf_matrix: TF-IDF matrix in CSR format
        percentile_threshold: Percentile threshold for considering a term score as "high"
        
    Returns:
        dict: Statistics about document uniqueness in the corpus
    """
    scores = []
    for i in range(tfidf_matrix.shape[0]):
        score = compute_document_uniqueness(i, tfidf_matrix, percentile_threshold)
        scores.append(score)
    
    return {
        'mean_uniqueness': np.mean(scores),
        'median_uniqueness': np.median(scores),
        'std_uniqueness': np.std(scores),
        'min_uniqueness': np.min(scores),
        'max_uniqueness': np.max(scores),
        'percentiles': {
            '25th': np.percentile(scores, 25),
            '75th': np.percentile(scores, 75),
            '90th': np.percentile(scores, 90)
        }
    }

# uniqueness_score = compute_document_uniqueness(
#     doc_idx=0, 
#     tfidf_matrix=tfidf_matrix,
#     percentile_threshold=95  # Consider terms in the top 5% as unusual
# )
# print(f"Document uniqueness score: {uniqueness_score:.3f}")

# # Get uniqueness statistics for the entire corpus
# corpus_stats = get_corpus_uniqueness_stats(tfidf_matrix)
# print("\nCorpus uniqueness statistics:")
# for key, value in corpus_stats.items():
#     if isinstance(value, dict):
#         print(f"\n{key}:")
#         for subkey, subvalue in value.items():
#             print(f"  {subkey}: {subvalue:.3f}")
#     else:
#         print(f"{key}: {value:.3f}")

In [1]:
%matplotlib inline
from typing import List, Dict, Tuple
from collections import Counter

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None) # don't truncate cell contents

from tqdm.notebook import tqdm
tqdm.pandas()

import numpy as np
import matplotlib.pyplot as plt

import spacy

# from spacy.tokenizer import Tokenizer
from spacy.lang.en import English
nlp = spacy.load('/srv/scratch2/kew/spacy_models/en_core_web_sm-2.3.1/en_core_web_sm/en_core_web_sm-2.3.1', disable=["tagger", "parser", "ner"])

from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer

import multiprocessing_utils as mp

/mnt/storage/clwork/users/kew/INSTALLS/anaconda3/envs/respondelligent/lib/python3.8/site-packages/tqdm/std.py:670: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [10]:
counts = ['cat zebra apple', 'cat zebra apple', 'banana lion apple elephant', 'elephant zebra elephant', 'elephant dog']

counts_lens = np.array([len(s.split()) for s in counts])
print(counts_lens)
transformer = TfidfVectorizer(smooth_idf=True, norm='l2')
tfidf = transformer.fit_transform(counts)
# print(tfidf)
# print(transformer.vocabulary_)
# print(dir(tfidf))
tfidf_array = tfidf.toarray()
# print(tfidf_array)
# print(tfidf_array.shape)


print(tfidf_array.sum(axis=1))
print(tfidf_array.sum(axis=1) / np.log(counts_lens))
# print(tfidf_array.sum(axis=1)

[3 3 4 3 2]
[1.72502774 1.72502774 1.9619827  1.34164079 1.38733127]
[1.57018792 1.57018792 1.41527135 1.22121407 2.00149594]
